# SentinelIQ — 06 SHAP Explainability
Generate per-anomaly SHAP attributions and MITRE ATT&CK mappings.

In [ ]:
import os, sys
repo = '/kaggle/working/SentinelIQ'
if os.path.exists(repo):
    !git -C /kaggle/working/SentinelIQ pull
else:
    !git clone https://github.com/hasan-rajab/SentinelIQ.git
%cd /kaggle/working/SentinelIQ
sys.path.insert(0, '/kaggle/working/SentinelIQ')
!pip install pyyaml joblib scikit-learn shap matplotlib -q


In [ ]:
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from ml.models.isolation_forest import SentinelIsolationForest
from ml.explainability.shap_explainer import SentinelShapExplainer
from ml.explainability.mitre_mapper import MitreMapper

with open('configs/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

def load_jsonl(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])

feature_cols = cfg['isolation_forest']['features']


In [ ]:
!python data/simulated/pipeline.py --duration 120 --anomaly-rate 0.08


## Load Model & Data

In [ ]:
metrics_df = load_jsonl('data/simulated/metrics.jsonl')
if_model = SentinelIsolationForest.load('ml/saved_models', name='isolation_forest_metrics')

X = metrics_df[feature_cols].fillna(0).values
X_scaled = if_model.scaler.transform(X)
y_true = metrics_df['is_anomaly'].astype(int).values

anomaly_idx = np.where(y_true == 1)[0]
normal_idx  = np.where(y_true == 0)[0]
print(f"Anomalies: {len(anomaly_idx)} | Normal: {len(normal_idx)}")


## Fit SHAP Explainer

In [ ]:
explainer = SentinelShapExplainer(model_type='isolation_forest', feature_cols=feature_cols)
explainer.fit(if_model.model, X_scaled[normal_idx[:200]])
shap_values = explainer.explain(X_scaled[anomaly_idx[:50]])
print(f"SHAP values shape: {shap_values.shape}")


## SHAP Summary Plot

In [ ]:
shap.summary_plot(
    shap_values,
    X_scaled[anomaly_idx[:50]],
    feature_names=feature_cols,
    show=True,
    plot_type="bar",
)


## SHAP Waterfall — Single Anomaly

In [ ]:
# Explain the highest-scoring anomaly
scores = if_model.score(metrics_df)
top_anomaly_idx = anomaly_idx[np.argmax(scores[anomaly_idx])]
top_anomaly = metrics_df.iloc[top_anomaly_idx]

print(f"Top anomaly record:")
print(f"  anomaly_type : {top_anomaly['anomaly_type']}")
print(f"  anomaly score: {scores[top_anomaly_idx]:.4f}")
for col in feature_cols:
    print(f"  {col:20}: {top_anomaly[col]}")

single_shap = explainer.explain(X_scaled[top_anomaly_idx:top_anomaly_idx+1])
attribution = explainer.explain_single(X_scaled[top_anomaly_idx])
print(f"
Top contributing features:")
for feat, val in list(attribution.items())[:5]:
    print(f"  {feat:20}: {val:+.6f}")


## MITRE ATT&CK Mapping

In [ ]:
mapper = MitreMapper()

anomalies = metrics_df[metrics_df['is_anomaly'] == True].copy()
anomaly_types = anomalies['anomaly_type'].unique()

print("MITRE ATT&CK Mappings for detected anomalies:
")
for atype in anomaly_types:
    m = mapper.map(atype)
    print(f"  [{atype}]")
    print(f"    Tactic    : {m.tactic} ({m.tactic_id})")
    print(f"    Technique : {m.technique} ({m.technique_id})")
    print(f"    Severity  : {m.severity.upper()}")
    print(f"    Action    : {m.recommended_action[:80]}")
    print()


## Full Anomaly Explanation

In [ ]:
from ml.explainability.shap_explainer import SentinelShapExplainer

# Build complete explanation for top anomaly
mitre = mapper.map_to_dict(top_anomaly['anomaly_type'])
fused_score = float(scores[top_anomaly_idx])
shap_attr = explainer.explain_single(X_scaled[top_anomaly_idx])

explanation = explainer.build_explanation(
    record=top_anomaly.to_dict(),
    shap_attribution=shap_attr,
    mitre_mapping=mitre,
    fused_score=fused_score,
)

print("=== ANOMALY EXPLANATION ===")
print(f"  Severity      : {explanation['severity'].upper()}")
print(f"  Fused Score   : {explanation['fused_score']}")
print(f"  MITRE Tactic  : {explanation['mitre_tactic']} ({explanation['mitre_tactic_id']})")
print(f"  Technique     : {explanation['mitre_technique']} ({explanation['mitre_technique_id']})")
print(f"  Top Features  : {explanation['top_features']}")
print(f"  Description   : {explanation['description']}")
print(f"  Action        : {explanation['recommended_action']}")
print()
print("=== NARRATIVE ===")
print(explainer.narrative(explanation))


## Tactic Distribution

In [ ]:
tactic_counts = {}
for _, row in anomalies.iterrows():
    m = mapper.map(row['anomaly_type'])
    tactic_counts[m.tactic] = tactic_counts.get(m.tactic, 0) + 1

plt.figure(figsize=(10, 4))
plt.barh(list(tactic_counts.keys()), list(tactic_counts.values()), color='#e74c3c')
plt.title('MITRE ATT&CK Tactic Distribution')
plt.xlabel('Count')
plt.tight_layout()
plt.show()
